In [ ]:
# ============================================
# CELLULE 1 — Vérification des dépendances
# ============================================

from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn
import xgboost

# Connexion à la session Snowflake
session = get_active_session()

print("✅ snowpark   : OK")
print(f"✅ pandas     : {pd.__version__}")
print(f"✅ numpy      : {np.__version__}")
print(f"✅ sklearn    : {sklearn.__version__}")
print(f"✅ matplotlib : {matplotlib.__version__}")
print(f"✅ seaborn    : {seaborn.__version__}")
print(f"✅ xgboost    : {xgboost.__version__}")
print(f"\n🔗 Connecté à Snowflake : {session.get_current_database()}.{session.get_current_schema()}")
print("\n🎉 Environnement prêt !")





In [ ]:
sql_statements = [
    """
    CREATE OR REPLACE FILE FORMAT HOUSE_JSON_FORMAT
      TYPE = 'JSON'
      STRIP_OUTER_ARRAY = TRUE
    """,
    """
    CREATE OR REPLACE STAGE HOUSE_STAGE
      URL = 's3://logbrain-datalake/datasets/house_price/'
      FILE_FORMAT = HOUSE_JSON_FORMAT
    """,
    """
    CREATE OR REPLACE TABLE HOUSE_PRICES_RAW (
      raw_data VARIANT
    )
    """,
    """
    COPY INTO HOUSE_PRICES_RAW
      FROM @HOUSE_STAGE/Housing_Price_Data.json
      FILE_FORMAT = HOUSE_JSON_FORMAT
    """
 ]

for stmt in sql_statements:
    session.sql(stmt).collect()

print("✅ Ingestion JSON terminée")
session.sql("SELECT COUNT(*) AS nb_lignes FROM HOUSE_PRICES_RAW").show()
session.sql("SELECT raw_data FROM HOUSE_PRICES_RAW LIMIT 3").show()

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE HOUSE_PRICES AS
SELECT
  raw_data:price::NUMBER             AS price,
  raw_data:area::NUMBER              AS area,
  raw_data:bedrooms::NUMBER          AS bedrooms,
  raw_data:bathrooms::NUMBER         AS bathrooms,
  raw_data:stories::NUMBER           AS stories,
  raw_data:mainroad::VARCHAR         AS mainroad,
  raw_data:guestroom::VARCHAR        AS guestroom,
  raw_data:basement::VARCHAR         AS basement,
  raw_data:hotwaterheating::VARCHAR  AS hotwaterheating,
  raw_data:airconditioning::VARCHAR  AS airconditioning,
  raw_data:parking::NUMBER           AS parking,
  raw_data:prefarea::VARCHAR         AS prefarea,
  raw_data:furnishingstatus::VARCHAR AS furnishingstatus
FROM HOUSE_PRICES_RAW
""").collect()

print("✅ Table HOUSE_PRICES créée")
session.sql("SELECT COUNT(*) AS nb_lignes FROM HOUSE_PRICES").show()
session.sql("SELECT * FROM HOUSE_PRICES LIMIT 5").show()

In [ ]:
# ============================================
# CELLULE 4 — Chargement & exploration
# ============================================

# Charger la table en DataFrame Snowpark
df = session.table("HOUSE_PRICES")

print(f"📊 Nombre de lignes  : {df.count()}")
print(f"📋 Nombre de colonnes: {len(df.columns)}")
print(f"\n📌 Colonnes : {df.columns}")

In [ ]:
# ============================================
# CELLULE 5 — Aperçu des données
# ============================================

# Aperçu des 5 premières lignes
print("=== 5 premières lignes ===")
df.show(5)

# Statistiques descriptives
print("\n=== Statistiques descriptives ===")
df.describe().show()

In [ ]:
# ============================================
# CELLULE 6 — Valeurs manquantes
# ============================================
from snowflake.snowpark.functions import col, count, when

# Compter les nulls par colonne
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

print("=== Valeurs manquantes par colonne ===")
null_counts.show()

In [ ]:
# ============================================
# CELLULE 7 — Distribution du prix
# ============================================
import matplotlib.pyplot as plt

# Convertir en Pandas pour visualiser
df_pd = df.to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(df_pd['PRICE'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution des Prix')
axes[0].set_xlabel('Prix')
axes[0].set_ylabel('Fréquence')

# Boxplot
axes[1].boxplot(df_pd['PRICE'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'))
axes[1].set_title('Boxplot des Prix')
axes[1].set_ylabel('Prix')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 8 — Matrice de corrélation
# ============================================
import seaborn as sns

# Sélectionner uniquement les colonnes numériques
num_cols = ['PRICE', 'AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'PARKING']
corr_matrix = df_pd[num_cols].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5)
plt.title('Matrice de Corrélation')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 9 — Variables catégorielles
# ============================================

cat_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT',
            'HOTWATERHEATING', 'AIRCONDITIONING',
            'PREFAREA', 'FURNISHINGSTATUS']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col_name in enumerate(cat_cols):
    counts = df_pd[col_name].value_counts()
    axes[i].bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{col_name}')
    axes[i].set_ylabel('Count')

# Masquer le dernier subplot vide
axes[-1].set_visible(False)

plt.suptitle('Distribution des Variables Catégorielles', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 10 — Encodage des variables
# ============================================
import pandas as pd

df_pd = df.to_pandas()

# Colonnes binaires (yes/no) → 1/0
binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT',
               'HOTWATERHEATING', 'AIRCONDITIONING', 'PREFAREA']

for col_name in binary_cols:
    df_pd[col_name] = df_pd[col_name].map({'yes': 1, 'no': 0})

# Colonne furnishingstatus → encodage ordinal
df_pd['FURNISHINGSTATUS'] = df_pd['FURNISHINGSTATUS'].map({
    'furnished': 2,
    'semi-furnished': 1,
    'unfurnished': 0
})

print("✅ Encodage terminé !")
print(df_pd.head())
print(f"\nTypes :\n{df_pd.dtypes}")

In [ ]:
# ============================================
# CELLULE 11 — Séparation features / cible
# ============================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Variable cible
y = df_pd['PRICE']

# Features (toutes les colonnes sauf PRICE)
X = df_pd.drop(columns=['PRICE'])

print(f"✅ Features X : {X.shape}")
print(f"✅ Cible y    : {y.shape}")
print(f"\n📌 Colonnes X : {list(X.columns)}")

In [ ]:
# ============================================
# CELLULE 12 — Train / Test split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train, 20% test
    random_state=42
)

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"✅ Taille X_train : {X_train_scaled.shape}")
print(f"✅ Taille X_test  : {X_test_scaled.shape}")
print(f"\n📊 Répartition :")
print(f"   Train : {len(X_train)} lignes ({len(X_train)/len(X)*100:.0f}%)")
print(f"   Test  : {len(X_test)} lignes  ({len(X_test)/len(X)*100:.0f}%)")

In [ ]:
# ============================================
# CELLULE 13 — Entraînement des modèles
# ============================================
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Définir les modèles
models = {
    "Linear Regression" : LinearRegression(),
    "Random Forest"     : RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost"           : XGBRegressor(n_estimators=100, random_state=42)
}

# Entraîner chaque modèle
trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f"✅ {name} entraîné !")

print("\n🎉 Tous les modèles sont entraînés !")

In [ ]:
# ============================================
# CELLULE 14 — Évaluation des modèles
# ============================================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    
    results.append({
        "Modèle" : name,
        "MAE"    : round(mae, 2),
        "RMSE"   : round(rmse, 2),
        "R²"     : round(r2, 4)
    })
    print(f"📊 {name}")
    print(f"   MAE  : {mae:,.0f}")
    print(f"   RMSE : {rmse:,.0f}")
    print(f"   R²   : {r2:.4f}\n")

# Tableau récapitulatif
results_df = pd.DataFrame(results)
print("=== Tableau comparatif ===")
print(results_df.to_string(index=False))

In [ ]:
# ============================================
# CELLULE 15 — Visualisation des performances
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['MAE', 'RMSE', 'R²']
colors  = ['#e74c3c', '#e67e22', '#2ecc71']

for i, metric in enumerate(metrics):
    axes[i].bar(results_df['Modèle'], results_df[metric],
                color=colors[i], edgecolor='white')
    axes[i].set_title(f'Comparaison — {metric}')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Comparaison des Modèles ML', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# CELLULE 16 — Optimisation avec GridSearchCV (pipeline complet)
# ============================================
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Paramètres réduits pour éviter la surcharge mémoire
param_grid = {
    'xgb__n_estimators': [100, 200],
    'xgb__max_depth': [3, 5],
    'xgb__learning_rate': [0.1, 0.2],
}

pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(random_state=42))
])

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=1,
    verbose=1
)

print("⏳ GridSearch en cours...")
grid_search.fit(X_train, y_train)

print(f"\n✅ Meilleurs paramètres : {grid_search.best_params_}")
print(f"✅ Meilleur R² (CV)     : {grid_search.best_score_:.4f}")

In [ ]:
# ============================================
# CELLULE 17 — Évaluation du meilleur modèle
# ============================================

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

mae_best  = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best   = r2_score(y_test, y_pred_best)

print("=== Modèle XGBoost Optimisé (Pipeline) ===")
print(f"   MAE  : {mae_best:,.0f}")
print(f"   RMSE : {rmse_best:,.0f}")
print(f"   R²   : {r2_best:.4f}")

# Comparaison avant/après
xgb_base = trained_models['XGBoost']
y_pred_base = xgb_base.predict(X_test_scaled)
r2_base = r2_score(y_test, y_pred_base)

print(f"\n📊 Comparaison :")
print(f"   XGBoost de base   → R² : {r2_base:.4f}")
print(f"   XGBoost optimisé  → R² : {r2_best:.4f}")
improvement = (r2_best - r2_base) * 100
print(f"   Amélioration      : +{improvement:.2f}%")

In [ ]:
# ============================================
# CELLULE 18 — Prédictions vs Réalité
# ============================================

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best, alpha=0.5, color='steelblue', edgecolors='white')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Prédiction parfaite')
plt.xlabel('Prix Réel')
plt.ylabel('Prix Prédit')
plt.title('XGBoost Optimisé — Prédictions vs Réalité')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 19 — Sauvegarde dans le Model Registry
# ============================================
from snowflake.ml.registry import Registry

# Créer le registry
registry = Registry(session=session)

# Enregistrer le meilleur modèle (pipeline complet: scaler + XGBoost)
model_ref = registry.log_model(
    model=best_model,
    model_name="house_price_xgboost",
    version_name="v1",
    comment="Pipeline XGBoost optimisé avec GridSearch",
    metrics={
        "r2": round(r2_best, 4),
        "mae": round(mae_best, 2),
        "rmse": round(rmse_best, 2)
    },
    sample_input_data=X_test.head(5)
)

print("✅ Modèle enregistré dans le Snowflake Model Registry !")
print("   Nom    : house_price_xgboost")
print("   Version: v1")
print(f"   R²     : {r2_best:.4f}")

In [ ]:
# ============================================
# CELLULE 22 — Vérification du registry
# ============================================

# Lister les modèles disponibles
models_list = registry.show_models()
print("=== Modèles dans le Registry ===")
print(models_list)

In [ ]:
# ============================================
# CELLULE 20 — Chargement du modèle depuis le Registry
# ============================================
from snowflake.ml.registry import Registry
import pandas as pd

# Charger le registry
registry = Registry(session=session)

# Récupérer le modèle
model_ref = registry.get_model("house_price_xgboost").version("v1")

print("✅ Modèle chargé depuis le registry !")

In [ ]:
# ============================================
# CELLULE 21 — Inférence corrigée
# ============================================

# Créer quelques nouvelles maisons à prédire
new_houses = pd.DataFrame({
    'AREA': [200, 150, 300],
    'BEDROOMS': [3, 2, 4],
    'BATHROOMS': [2, 1, 3],
    'STORIES': [2, 1, 3],
    'MAINROAD': [1, 1, 0],
    'GUESTROOM': [0, 0, 1],
    'BASEMENT': [1, 0, 1],
    'HOTWATERHEATING': [0, 0, 1],
    'AIRCONDITIONING': [1, 0, 1],
    'PARKING': [2, 1, 3],
    'PREFAREA': [1, 0, 1],
    'FURNISHINGSTATUS': [2, 0, 1]
}).astype('int16')

# Le modèle enregistré contient déjà le scaler
predictions = model_ref.run(new_houses, function_name="predict")

print("=== Prédictions de prix ===")
for i, pred in enumerate(predictions['output_feature_0']):
    print(f"   Maison {i+1} : {pred:,.0f} €")

In [ ]:
# ============================================
# CELLULE 23 — Stocker les résultats dans Snowflake
# ============================================

# Ajouter les prédictions au dataframe
new_houses['PREDICTED_PRICE'] = predictions['output_feature_0'].values

# Sauvegarder dans une table Snowflake
session.write_pandas(
    new_houses,
    table_name    = "HOUSE_PREDICTIONS",
    auto_create_table = True,
    overwrite     = True
)

print("✅ Prédictions sauvegardées dans la table HOUSE_PREDICTIONS !")

# Vérifier
session.table("HOUSE_PREDICTIONS").show()

In [ ]:
# ============================================
# CELLULE 24 — Application Streamlit (UI amelioree)
# ============================================
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

st.set_page_config(page_title="Estimateur Immobilier", page_icon="🏠", layout="wide")

st.markdown("""
<style>
:root {
  --bg-soft: #f8f7f2;
  --card-bg: #ffffff;
  --ink: #1f2937;
  --accent: #0f766e;
  --accent-2: #f59e0b;
}

div.block-container {
  padding-top: 1.4rem;
  padding-bottom: 2rem;
}

html, body, [class*="css"] {
  font-family: "Trebuchet MS", "Segoe UI", sans-serif;
}

.custom-hero {
  background: linear-gradient(120deg, #0f766e 0%, #14b8a6 45%, #f59e0b 100%);
  color: white;
  border-radius: 20px;
  padding: 1.2rem 1.4rem;
  margin-bottom: 1rem;
  box-shadow: 0 10px 28px rgba(0, 0, 0, 0.12);
}

.custom-card {
  background: var(--card-bg);
  border: 1px solid #e5e7eb;
  border-left: 6px solid var(--accent);
  border-radius: 14px;
  padding: 0.9rem 1rem;
  margin: 0.4rem 0 0.8rem 0;
  box-shadow: 0 6px 18px rgba(15, 118, 110, 0.08);
}

.label-muted {
  color: #6b7280;
  font-size: 0.9rem;
}
</style>
""", unsafe_allow_html=True)

try:
    session = get_active_session()
except Exception:
    st.error("Session Snowflake introuvable. Lance cette app dans Snowflake Streamlit.")
    st.stop()

st.markdown(
    """
    <div class='custom-hero'>
      <h2 style='margin:0;'>Estimateur de Prix Immobilier</h2>
      <p style='margin:0.35rem 0 0 0;'>
        Renseigne les caracteristiques d'une maison pour obtenir une estimation immediate du prix.
      </p>
    </div>
    """,
    unsafe_allow_html=True,
)

col_a, col_b = st.columns([1.15, 1], gap="large")

with col_a:
    st.markdown("### Caracteristiques principales")

    area = st.slider("Surface (m2)", min_value=50, max_value=1000, value=150, step=5)
    bedrooms = st.slider("Chambres", min_value=1, max_value=10, value=3)
    bathrooms = st.slider("Salles de bain", min_value=1, max_value=5, value=2)
    stories = st.slider("Etages", min_value=1, max_value=5, value=2)
    parking = st.slider("Places de parking", min_value=0, max_value=5, value=1)

    st.markdown("### Confort et localisation")
    c1, c2, c3 = st.columns(3)

    with c1:
        mainroad = st.selectbox("Route principale", ["yes", "no"])
        guestroom = st.selectbox("Chambre d'amis", ["yes", "no"])
        basement = st.selectbox("Sous-sol", ["yes", "no"])

    with c2:
        hotwater = st.selectbox("Chauffage eau chaude", ["yes", "no"])
        aircon = st.selectbox("Climatisation", ["yes", "no"])
        prefarea = st.selectbox("Zone privilegiee", ["yes", "no"])

    with c3:
        furnished = st.selectbox(
            "Ameublement",
            ["furnished", "semi-furnished", "unfurnished"]
        )

with col_b:
    st.markdown("### Resume de la saisie")

    def badge(value):
        return "Oui" if value == "yes" else "Non"

    st.markdown(
        f"""
        <div class='custom-card'>
          <div><strong>Surface:</strong> {area} m2</div>
          <div><strong>Configuration:</strong> {bedrooms} ch, {bathrooms} sdb, {stories} etages</div>
          <div><strong>Stationnement:</strong> {parking} place(s)</div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    st.markdown(
        f"""
        <div class='custom-card'>
          <div class='label-muted'>Confort et emplacement</div>
          <div>Route principale: <strong>{badge(mainroad)}</strong></div>
          <div>Chambre d'amis: <strong>{badge(guestroom)}</strong></div>
          <div>Sous-sol: <strong>{badge(basement)}</strong></div>
          <div>Chauffage eau chaude: <strong>{badge(hotwater)}</strong></div>
          <div>Climatisation: <strong>{badge(aircon)}</strong></div>
          <div>Zone privilegiee: <strong>{badge(prefarea)}</strong></div>
          <div>Ameublement: <strong>{furnished}</strong></div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    comfort_score = sum([
        1 if mainroad == "yes" else 0,
        1 if guestroom == "yes" else 0,
        1 if basement == "yes" else 0,
        1 if hotwater == "yes" else 0,
        1 if aircon == "yes" else 0,
        1 if prefarea == "yes" else 0,
    ])
    st.progress(comfort_score / 6, text=f"Score confort: {comfort_score}/6")

def encode(val):
    return 1 if val == "yes" else 0

furnish_map = {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}

if st.button("Estimer le prix", type="primary", use_container_width=True):
    input_data = pd.DataFrame([{
        "AREA": area,
        "BEDROOMS": bedrooms,
        "BATHROOMS": bathrooms,
        "STORIES": stories,
        "MAINROAD": encode(mainroad),
        "GUESTROOM": encode(guestroom),
        "BASEMENT": encode(basement),
        "HOTWATERHEATING": encode(hotwater),
        "AIRCONDITIONING": encode(aircon),
        "PARKING": parking,
        "PREFAREA": encode(prefarea),
        "FURNISHINGSTATUS": furnish_map[furnished],
    }]).astype("int16")

    with st.spinner("Prediction en cours..."):
        registry = Registry(session=session)
        model_ref = registry.get_model("house_price_xgboost").version("v1")
        result = model_ref.run(input_data, function_name="predict")
        price = float(result["output_feature_0"].values[0])

    ppsm = price / max(area, 1)

    c1, c2 = st.columns(2)
    c1.metric("Prix estime", f"{price:,.0f} EUR")
    c2.metric("Prix au m2", f"{ppsm:,.0f} EUR/m2")

    st.markdown("### Donnees envoyees au modele")
    st.dataframe(input_data, use_container_width=True)
    st.success("Estimation terminee avec succes.")